In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv

from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

llm = GoogleGenerativeAI(
    model = "gemini-2.5-flash-lite"
)

In [3]:
class JokeState(TypedDict):
    
    topic: str 
    joke: str 
    explaination: str 

In [4]:
def generate_joke(state:JokeState):

    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt)

    return {'joke': response}

In [5]:
def generate_explaination(state:JokeState):
    prompt = f"write an explaination for the joke - {state['joke']}"

    response = llm.invoke(prompt)

    return {"explaination":response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explaination" , generate_explaination)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke" , "generate_explaination")
graph.add_edge("generate_explaination", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer= checkpointer)

In [7]:
config = {"configurable": {"thread_id" : "1"}}

workflow.invoke({"topic": "pizza"}, config=config)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!',
 'explaination': 'This joke is a classic example of a **pun**, which means it uses words that sound alike but have different meanings to create humor. Let\'s break down why it\'s funny:\n\n*   **"Crust"**:\n    *   **Literal meaning (pizza):** The outer edge of the pizza, the part you often eat first or last.\n    *   **Figurative meaning (comedian):** In the context of a comedian, "crust" can be a slang term for **boldness, confidence, or a bit of cheekiness**. A comedian with "crust" isn\'t afraid to take risks or be a little provocative.\n\n*   **"Dough"**:\n    *   **Literal meaning (pizza):** The uncooked mixture of flour, water, and yeast that forms the base of a pizza.\n    *   **Figurative meaning (comedian):** "Dough" is also slang for **money**. Comedians, like anyone with a job, are often looking to "deliver the dough" – meaning th

In [8]:
workflow.get_state(config)
# will be stored even after end of graph (this one is stored in RAM)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!', 'explaination': 'This joke is a classic example of a **pun**, which means it uses words that sound alike but have different meanings to create humor. Let\'s break down why it\'s funny:\n\n*   **"Crust"**:\n    *   **Literal meaning (pizza):** The outer edge of the pizza, the part you often eat first or last.\n    *   **Figurative meaning (comedian):** In the context of a comedian, "crust" can be a slang term for **boldness, confidence, or a bit of cheekiness**. A comedian with "crust" isn\'t afraid to take risks or be a little provocative.\n\n*   **"Dough"**:\n    *   **Literal meaning (pizza):** The uncooked mixture of flour, water, and yeast that forms the base of a pizza.\n    *   **Figurative meaning (comedian):** "Dough" is also slang for **money**. Comedians, like anyone with a job, are often looking to "deliver the 

In [9]:
# history (checkpoints -> at each nodes)

list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!', 'explaination': 'This joke is a classic example of a **pun**, which means it uses words that sound alike but have different meanings to create humor. Let\'s break down why it\'s funny:\n\n*   **"Crust"**:\n    *   **Literal meaning (pizza):** The outer edge of the pizza, the part you often eat first or last.\n    *   **Figurative meaning (comedian):** In the context of a comedian, "crust" can be a slang term for **boldness, confidence, or a bit of cheekiness**. A comedian with "crust" isn\'t afraid to take risks or be a little provocative.\n\n*   **"Dough"**:\n    *   **Literal meaning (pizza):** The uncooked mixture of flour, water, and yeast that forms the base of a pizza.\n    *   **Figurative meaning (comedian):** "Dough" is also slang for **money**. Comedians, like anyone with a job, are often looking to "deliver the

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the meatball?\n\nBecause he found out she was seeing other noodles!',
 'explaination': 'This joke is a play on words and uses a common idiom. Here\'s the breakdown:\n\n*   **The Setup:** "Why did the spaghetti break up with the meatball?" This sets up a classic "why did the X do Y" joke format, leading you to expect a punchline.\n\n*   **The Punchline:** "Because he found out she was seeing other noodles!"\n\n*   **The Wordplay/Pun:** The humor comes from the double meaning of "noodles."\n\n    *   **Literal Meaning (Food):** Spaghetti itself is a type of noodle. So, in the literal sense, the spaghetti (which is a noodle) is upset that its partner (the meatball) is "seeing other noodles" – meaning other kinds of pasta like penne, fusilli, or even other spaghetti strands. This is absurd because spaghetti *is* a noodle.\n\n    *   **Figurative Meaning (Slang):** In informal slang, "noodles" can be a humorous or slightly der

In [11]:
workflow.get_state(config)
# there are 2 invokes , we can access both of those using thread id 

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!', 'explaination': 'This joke is a classic example of a **pun**, which means it uses words that sound alike but have different meanings to create humor. Let\'s break down why it\'s funny:\n\n*   **"Crust"**:\n    *   **Literal meaning (pizza):** The outer edge of the pizza, the part you often eat first or last.\n    *   **Figurative meaning (comedian):** In the context of a comedian, "crust" can be a slang term for **boldness, confidence, or a bit of cheekiness**. A comedian with "crust" isn\'t afraid to take risks or be a little provocative.\n\n*   **"Dough"**:\n    *   **Literal meaning (pizza):** The uncooked mixture of flour, water, and yeast that forms the base of a pizza.\n    *   **Figurative meaning (comedian):** "Dough" is also slang for **money**. Comedians, like anyone with a job, are often looking to "deliver the 

In [16]:
list(workflow.get_state_history(config))
# history for 1st invoke 

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!', 'explaination': 'This joke is a classic example of a **pun**, which means it uses words that sound alike but have different meanings to create humor. Let\'s break down why it\'s funny:\n\n*   **"Crust"**:\n    *   **Literal meaning (pizza):** The outer edge of the pizza, the part you often eat first or last.\n    *   **Figurative meaning (comedian):** In the context of a comedian, "crust" can be a slang term for **boldness, confidence, or a bit of cheekiness**. A comedian with "crust" isn\'t afraid to take risks or be a little provocative.\n\n*   **"Dough"**:\n    *   **Literal meaning (pizza):** The uncooked mixture of flour, water, and yeast that forms the base of a pizza.\n    *   **Figurative meaning (comedian):** "Dough" is also slang for **money**. Comedians, like anyone with a job, are often looking to "deliver the

## Time Travel

In [18]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1187f3-3638-6399-8000-8dae62fb5953"}})
# going to intermediate state using checkpoints(pasted it from history)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1187f3-3638-6399-8000-8dae62fb5953'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-05T10:36:42.726901+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1187f3-3633-6577-bfff-4e86ec70b179'}}, tasks=(PregelTask(id='c752c1d5-4305-f380-0810-5a4e62bd8cf8', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza get a job as a comedian?\n\nBecause it had a great **crust** and always delivered the **dough**!'}),), interrupts=())

In [19]:
# rerunning the code from that checkpoints 

workflow.invoke(None ,{"configurable": {"thread_id": "1","checkpoint_id": "1f1187f3-3638-6399-8000-8dae62fb5953"}} )

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job as a therapist?\n\nBecause it was great at helping people solve their **dough-mas**!',
 'explaination': 'This joke is a play on words, specifically using a pun. Here\'s the breakdown:\n\n* **"Pizza" and "Therapist":** The setup creates an absurd image of a pizza working as a therapist. This immediately signals that the punchline will likely be humorous and unexpected.\n\n* **"Dough-mas":** This is the core of the pun.\n    * **"Dough"** is the primary ingredient in pizza. It\'s what makes up the crust.\n    * **"Dramas"** is a common term for serious problems, emotional turmoil, or conflicts that people often discuss with therapists.\n\n* **The Connection:** The joke cleverly substitutes "dough" for the first syllable of "dramas." The sound is very similar, making it a homophone (or near-homophone).\n\n* **The Humor:** The humor comes from the unexpected and silly connection. A therapist\'s job is to help people work through thei

In [20]:
list(workflow.get_state_history(config))


[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a therapist?\n\nBecause it was great at helping people solve their **dough-mas**!', 'explaination': 'This joke is a play on words, specifically using a pun. Here\'s the breakdown:\n\n* **"Pizza" and "Therapist":** The setup creates an absurd image of a pizza working as a therapist. This immediately signals that the punchline will likely be humorous and unexpected.\n\n* **"Dough-mas":** This is the core of the pun.\n    * **"Dough"** is the primary ingredient in pizza. It\'s what makes up the crust.\n    * **"Dramas"** is a common term for serious problems, emotional turmoil, or conflicts that people often discuss with therapists.\n\n* **The Connection:** The joke cleverly substitutes "dough" for the first syllable of "dramas." The sound is very similar, making it a homophone (or near-homophone).\n\n* **The Humor:** The humor comes from the unexpected and silly connection. A therapist\'s job is to help peop

In [23]:
# updating topic at middle of "thread 1 invoked"ArithmeticError

workflow.update_state({"configurable":{"thread_id": "1" , "checkpoint_id": '1f1187f3-3638-6399-8000-8dae62fb5953', "checkpoint_ns": ""}} , {"topic": "samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f11880a-ed1f-697b-8001-3c77e300ea6c'}}

In [25]:
list(workflow.get_state_history(config))


[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11880a-ed1f-697b-8001-3c77e300ea6c'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-05T10:47:19.307301+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1187f3-3638-6399-8000-8dae62fb5953'}}, tasks=(PregelTask(id='2e28b85c-e2e5-8c38-3e8b-1ee35a0ecdd0', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job as a therapist?\n\nBecause it was great at helping people solve their **dough-mas**!', 'explaination': 'This joke is a play on words, specifically using a pun. Here\'s the breakdown:\n\n* **"Pizza" and "Therapist":** The setup creates an absurd image of a pizza working as a therapist. This immediately

In [26]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": '1f11880a-ed1f-697b-8001-3c77e300ea6c'}})


{'topic': 'samosa',
 'joke': 'Why did the samosa break up with the pakora?\n\nBecause they had too many *filling* issues!',
 'explaination': 'This joke plays on a **pun**. Here\'s the breakdown:\n\n*   **Samosa and Pakora:** These are both popular Indian fried snacks.\n    *   A **samosa** is typically a triangular pastry filled with spiced potatoes, peas, and sometimes meat.\n    *   A **pakora** is a fritter made by deep-frying vegetables (like onions, potatoes, spinach) or sometimes meat, coated in a spiced batter.\n\n*   **"Filling" Issues:** This is the key to the pun.\n    *   **Literal Meaning:** Samosas are known for their **filling** (the spiced potato and pea mixture inside). If there were problems with the filling (e.g., it was too spicy, not cooked enough, etc.), that would be a literal "filling issue" for the samosa.\n    *   **Figurative Meaning:** In relationships, "filling issues" can refer to **emotional baggage, unresolved problems, or things that a person is carrying

In [27]:
list(workflow.get_state_history(config))


[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa break up with the pakora?\n\nBecause they had too many *filling* issues!', 'explaination': 'This joke plays on a **pun**. Here\'s the breakdown:\n\n*   **Samosa and Pakora:** These are both popular Indian fried snacks.\n    *   A **samosa** is typically a triangular pastry filled with spiced potatoes, peas, and sometimes meat.\n    *   A **pakora** is a fritter made by deep-frying vegetables (like onions, potatoes, spinach) or sometimes meat, coated in a spiced batter.\n\n*   **"Filling" Issues:** This is the key to the pun.\n    *   **Literal Meaning:** Samosas are known for their **filling** (the spiced potato and pea mixture inside). If there were problems with the filling (e.g., it was too spicy, not cooked enough, etc.), that would be a literal "filling issue" for the samosa.\n    *   **Figurative Meaning:** In relationships, "filling issues" can refer to **emotional baggage, unresolved problems, or things that 

### Fault Tolerance using persistance


In [ ]:
class CrashState(TypedDict):

    input: str 
    step1 : str 
    step2 : str 
    step3 : str 

In [15]:
import time

In [16]:
def step_1(state: CrashState):
    print("Step1 executed")
    return {"step1":"done" }

In [17]:
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [18]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2" , step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1" , "step_2")
builder.add_edge("step_2" , "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).") 

▶️ Running graph: Please manually interrupt during Step 2...
Step1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))